# Connect 4 AlphaZero — Training Analysis

Visualize training metrics from a completed or in-progress training run.
Point `LOG_FILE` at your training log (or `CHECKPOINTS_DIR` at your checkpoints folder).

In [ ]:
import re
import json
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# --- Configuration ---
LOG_FILE = '../logs/training.log'          # Path to training log file
CHECKPOINTS_DIR = '../checkpoints'         # Path to checkpoints directory

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
def parse_training_log(log_path: str) -> dict:
    """Parse training log file into per-iteration metrics."""
    metrics = {
        'iteration': [],
        'policy_loss': [],
        'value_loss': [],
        'total_loss': [],
        'arena_wins': [],
        'arena_losses': [],
        'arena_draws': [],
        'win_rate': [],
        'accepted': [],
        'buffer_size': [],
    }

    if not Path(log_path).exists():
        print(f"Log file not found: {log_path}")
        return metrics

    with open(log_path) as f:
        text = f.read()

    iter_blocks = re.split(r'=== Iteration (\d+)', text)

    for i in range(1, len(iter_blocks), 2):
        iter_num = int(iter_blocks[i])
        block = iter_blocks[i + 1]

        metrics['iteration'].append(iter_num)

        # Buffer size
        m = re.search(r'buffer size[:\s]+(\d+)', block, re.IGNORECASE)
        metrics['buffer_size'].append(int(m.group(1)) if m else None)

        # Losses
        m = re.search(r'policy[:\s]+([\d.]+)\s+value[:\s]+([\d.]+)\s+total[:\s]+([\d.]+)', block, re.IGNORECASE)
        if m:
            metrics['policy_loss'].append(float(m.group(1)))
            metrics['value_loss'].append(float(m.group(2)))
            metrics['total_loss'].append(float(m.group(3)))
        else:
            metrics['policy_loss'].append(None)
            metrics['value_loss'].append(None)
            metrics['total_loss'].append(None)

        # Arena results
        m = re.search(r'wins[:\s]+(\d+)\s+losses[:\s]+(\d+)\s+draws[:\s]+(\d+)\s+win_rate[:\s]+([\d.]+)', block, re.IGNORECASE)
        if m:
            metrics['arena_wins'].append(int(m.group(1)))
            metrics['arena_losses'].append(int(m.group(2)))
            metrics['arena_draws'].append(int(m.group(3)))
            metrics['win_rate'].append(float(m.group(4)))
        else:
            metrics['arena_wins'].append(None)
            metrics['arena_losses'].append(None)
            metrics['arena_draws'].append(None)
            metrics['win_rate'].append(None)

        # Accepted/rejected
        metrics['accepted'].append('Accepted' in block)

    print(f"Parsed {len(metrics['iteration'])} iterations from log.")
    return metrics


metrics = parse_training_log(LOG_FILE)

In [ ]:
# --- Plot 1: Loss Curves ---
iters = metrics['iteration']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, key, title, color in [
    (axes[0], 'policy_loss', 'Policy Loss', 'steelblue'),
    (axes[1], 'value_loss', 'Value Loss', 'darkorange'),
    (axes[2], 'total_loss', 'Total Loss', 'forestgreen'),
]:
    vals = [v for v in metrics[key] if v is not None]
    its = [i for i, v in zip(iters, metrics[key]) if v is not None]
    ax.plot(its, vals, 'o-', color=color, linewidth=2, markersize=5)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')

plt.suptitle('Training Loss Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../logs/loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: logs/loss_curves.png")

In [ ]:
# --- Plot 2: Arena Win Rate ---
win_rates = [(i, w, a) for i, w, a in zip(iters, metrics['win_rate'], metrics['accepted']) if w is not None]

if win_rates:
    fig, ax = plt.subplots(figsize=(12, 4))

    accepted_its = [i for i, w, a in win_rates if a]
    accepted_wr  = [w for i, w, a in win_rates if a]
    rejected_its = [i for i, w, a in win_rates if not a]
    rejected_wr  = [w for i, w, a in win_rates if not a]

    ax.scatter(accepted_its, accepted_wr, color='forestgreen', s=80, zorder=3, label='Accepted')
    ax.scatter(rejected_its, rejected_wr, color='crimson', s=80, marker='x', zorder=3, label='Rejected')
    ax.plot([i for i, w, a in win_rates], [w for i, w, a in win_rates], 'k-', alpha=0.3, linewidth=1)

    # 55% acceptance threshold
    ax.axhline(0.55, color='darkorange', linestyle='--', linewidth=1.5, label='Acceptance threshold (55%)')
    ax.axhline(0.50, color='gray', linestyle=':', linewidth=1, label='Even (50%)')

    ax.set_title('Arena Win Rate per Iteration', fontsize=13, fontweight='bold')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Win Rate')
    ax.set_ylim(0, 1)
    ax.legend()

    plt.tight_layout()
    plt.savefig('../logs/arena_win_rate.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: logs/arena_win_rate.png")
    
    total = len(win_rates)
    accepted = sum(1 for _, _, a in win_rates if a)
    print(f"\nAccepted {accepted}/{total} iterations ({100*accepted/total:.0f}%)")
else:
    print("No arena data found in log.")

In [ ]:
# --- Plot 3: Replay Buffer Size ---
buf_data = [(i, s) for i, s in zip(iters, metrics['buffer_size']) if s is not None]

if buf_data:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot([i for i, s in buf_data], [s for i, s in buf_data], 'o-', color='purple', linewidth=2, markersize=5)
    ax.set_title('Replay Buffer Size', fontsize=13, fontweight='bold')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Samples in Buffer')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Value Head Collapse Diagnostic ---
# Load the best checkpoint and check value prediction distribution
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
from src.utils.config import Config
from src.neural_net.model import Connect4Net
from src.game.board import Connect4Board

best_ckpt = Path(CHECKPOINTS_DIR) / 'best_model.pt'

if best_ckpt.exists():
    ckpt = torch.load(best_ckpt, weights_only=False)
    from src.utils.config import ModelConfig
    model_cfg = ModelConfig(
        num_residual_blocks=ckpt['num_blocks'],
        num_filters=ckpt['num_filters']
    )
    model = Connect4Net(model_cfg)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    # Generate random board positions and collect value predictions
    values = []
    board = Connect4Board()
    np.random.seed(42)
    for _ in range(200):
        b = Connect4Board()
        # Random moves (0-30 moves)
        n_moves = np.random.randint(0, 20)
        for _ in range(n_moves):
            if b.is_terminal():
                break
            legal = b.get_legal_moves()
            b = b.make_move(np.random.choice(legal))
        if not b.is_terminal():
            enc = b.encode()
            state = torch.tensor(enc.tolist(), dtype=torch.float32).unsqueeze(0)
            with torch.no_grad():
                _, v = model(state)
            values.append(v.item())

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(values, bins=30, color='steelblue', edgecolor='white', linewidth=0.5)
    ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='v=0 (draw prediction)')
    ax.set_title('Value Head Prediction Distribution (200 random positions)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Value')
    ax.set_ylabel('Count')
    ax.legend()
    plt.tight_layout()
    plt.show()

    mean_v = np.mean(np.abs(values))
    print(f"Mean |value|: {mean_v:.3f}")
    if mean_v < 0.15:
        print("WARNING: Value predictions clustered near 0 — possible value head collapse.")
        print("Try restarting training with learning_rate: 5.0e-4")
    else:
        print("Value head looks healthy.")
else:
    print(f"No best_model.pt found at {best_ckpt}")

In [ ]:
# --- Summary Table ---
print(f"{'Iter':>5}  {'Policy':>8}  {'Value':>8}  {'Total':>8}  {'WinRate':>8}  {'Status':>10}")
print('-' * 65)

for i, (pl, vl, tl, wr, acc) in enumerate(zip(
    metrics['policy_loss'], metrics['value_loss'],
    metrics['total_loss'], metrics['win_rate'], metrics['accepted']
)):
    pl_s  = f"{pl:.4f}"  if pl  is not None else '      -'
    vl_s  = f"{vl:.4f}"  if vl  is not None else '      -'
    tl_s  = f"{tl:.4f}"  if tl  is not None else '      -'
    wr_s  = f"{wr:.3f}"  if wr  is not None else '      -'
    st_s  = 'ACCEPTED' if acc else ('rejected' if wr is not None else 'first')
    print(f"{metrics['iteration'][i]:>5}  {pl_s:>8}  {vl_s:>8}  {tl_s:>8}  {wr_s:>8}  {st_s:>10}")